In [ ]:
import os
import numpy as np
import torch

from run_utils import initialize_DDT, parse_class_labels, tensor_to_image

In [2]:
## Download DDT-XL checkpoint if not exists
if os.path.exists("model.ckpt"):
    print("Checkpoint found!")
else:
    !wget https://huggingface.co/MCG-NJU/DDT-XL-22en6de-R512/resolve/main/model.ckpt


## GPU-memory management
!export PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True

Checkpoint found!


In [ ]:
# Settings
config_path = "configs/repa_improved_ddt_xlen22de6_512.yaml"
checkpoint_path = "model.ckpt"
output_dir = "../outputs_images"
classes = ["tiger cat"]
resolution = 512
num_images = 1
seed = 1234

# Prepare environment
os.makedirs(output_dir, exist_ok=True)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cpu


In [ ]:
vae, denoiser, conditioner, sampler = initialize_DDT(device=device)

In [ ]:
# Process class names
label_map = parse_class_labels()
valid_classes = [cls for cls in classes if cls.lower() in label_map]

/home/c/cimsir/PracVRL/.venv/lib/python3.12/site-packages/torch/cuda/__init__.py:716: UserWarning: Can't initialize NVML
  warnings.warn("Can't initialize NVML")
current sampler is ODE sampler, but w_scheduler is enabled


In [8]:
for class_name in valid_classes:
    class_id = label_map[class_name.lower()]
    print(f"\nGenerating images for: {class_name} (ID: {class_id})")
    
    for i in range(num_images):
        img_seed = seed + i * 10
        generator = torch.Generator().manual_seed(img_seed)
        noise = torch.randn((1, 4, resolution // 8, resolution // 8), generator=generator).to(device)

        with torch.no_grad():
            cond, uncond = conditioner([class_id])
            output = sampler(denoiser, noise, cond.to(device), uncond.to(device))
            if output is None:
                print(f"Sampler failed for {class_name}, seed {img_seed}")
                continue
            decoded = vae.decode(output.to(device))
            if decoded is None:
                print(f"Decoding failed for {class_name}, seed {img_seed}")
                continue

            img_tensor = tensor_to_image(decoded.cpu())[0].permute(1, 2, 0).numpy()
            img_tensor = img_tensor[:, :, :3] if img_tensor.shape[2] > 3 else img_tensor

            img = Image.fromarray(img_tensor)
            fname = f"{class_name.replace(' ', '_')}_seed{img_seed}.png"
            img.save(os.path.join(output_dir, fname))
            print(f"Saved: {fname}")

print("\nAll done.")


Generating images for: tiger cat (ID: 282)


RuntimeError: No CUDA GPUs are available